# Celcomen on `crc_232`

[Celcomen](https://github.com/Teichlab/celcomen) (Megas et al., 2025) learns one global gene-gene
interaction matrix (`G2G`, inter-cellular) plus an intracellular one from a spatial graph, then uses
Simcomen to impose a perturbation as an initial condition and relax the whole expression field.

This notebook runs it, as its authors prescribe
([`Tutorial_Celcomen_on_Xenium.ipynb`](https://github.com/Teichlab/celcomen/blob/main/Tutorial_Celcomen_on_Xenium.ipynb)),
on the setting our leave-one-cell-type-out benchmark uses: slide `crc_232`, 2000 `seurat_v3` HVGs,
`(Myeloid, CRC)` held out as graph nodes. Hyperparameters are the tutorial's (`k=6`, `zmft=1e-1`,
`lr=1e-1`, 200 epochs, `normalize_total(1e6)` + `log1p`).

It does not scale to this slide. Celcomen's objective contains
`trace(msg @ gex.T)` (`celcomen/training_plan/train.py:60`), which materialises a dense
**cell-by-cell** matrix, quadratic in the number of cells. Section 3 runs the released training
function unchanged and captures the resulting allocation failure.

Run in the `celcomen` kernel (`environments/celcomen.yml`); it shares nothing with the `cellina` stack.

In [1]:
# --- Config ---------------------------------------------------------------
import time
import traceback

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from sklearn.neighbors import kneighbors_graph

if torch.cuda.is_available():                     # shared box: take the emptiest GPU
    free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
    DEVICE = f"cuda:{int(np.argmax(free))}"
else:
    DEVICE = "cpu"

ADATA_PATH = "../../../data/crc_wt_cosmx/crc_232.h5ad"
ZENODO_URL = "https://zenodo.org/records/15574384/files/232.h5ad?download=1"

# benchmark setting, copied from the Cellina LOO run
HOLDOUT_CT, LABELS_KEY, DOMAINS_KEY = "Myeloid", "coarse_type", "typ"
N_HVG = 2000
# Celcomen tutorial hyperparameters
K_NN, ZMFT, SEED, LR, EPOCHS, TARGET_SUM = 6, 1e-1, 0, 1e-1, 200, 1e6

np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"device={DEVICE}  torch={torch.__version__}")

device=cuda:1  torch=2.5.1+cu121


## 1. Slide `crc_232` in the benchmark's gene space

In [2]:
# --- Load + preprocess ----------------------------------------------------
label_to_coarse = {
    "epi1": "Epithelial", "epi2": "Epithelial", "epi3": "Epithelial", "epi4": "Epithelial",
    "fib1": "Fibroblast", "fib2": "Fibroblast", "EC": "Endothelial", "SMC": "Smooth_muscle",
    "BC": "B_cell", "PC_IgA": "Plasma_cell", "PC_IgG": "Plasma_cell", "PC_IgM": "Plasma_cell",
    "TC": "T_cell", "mye1": "Myeloid", "mye2": "Myeloid", "mast": "Mast_cell",
}

adata = sc.read(ADATA_PATH, backup_url=ZENODO_URL)
adata.obs_names_make_unique()
adata.obs[LABELS_KEY] = adata.obs["ist"].map(label_to_coarse)
adata = adata[~adata.obs[DOMAINS_KEY].isna()]
adata = adata[~adata.obs[LABELS_KEY].isna()].copy()
sc.pp.filter_cells(adata, min_counts=3)
sc.pp.filter_genes(adata, min_counts=3)
adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, layer="counts", flavor="seurat_v3", n_top_genes=N_HVG, subset=True)
adata.obsm["spatial"] = adata.obs[["CenterX_global_px", "CenterY_global_px"]].values.astype("float64")
adata

AnnData object with n_obs × n_vars = 111989 × 2000
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68_CK8_18', 'Max.CD68_CK8_18', 'Mean.CD298_B2M', 'Max.CD298_B2M', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'cell_id', 'Dash', 'ISH.concentration', 'Panel', 'Run_Tissue_name', 'Run_name', 'assay_type', 'dualfiles', 'tissue', 'version', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_0.8', 'falsecode_quantile

## 2. Held-out nodes and Celcomen's expression representation

Celcomen has no cell-type labels and learns a single global interaction matrix, so holding out
`(Myeloid, CRC)` can only mean dropping those cells as graph nodes: they then neither inform `G2G`
nor act as anyone's neighbour.

Expression is log-normalised as the tutorial does, then row-normalised. Note the released code is
inconsistent here: `celcomen/datareaders/datareader.py` divides by `||x||**2`
(`x / x.pow(2).sum(1)`, no square root) while the tutorial's own Simcomen cell divides by `||x||`,
which is what the model's spherical geometry assumes. The released `||x||**2` form is used below,
since this section runs the released training path.

In [3]:
# --- Held-out nodes + row-normalised expression ---------------------------
is_crc = adata.obs[DOMAINS_KEY].str.contains("CRC").values
is_mye = (adata.obs[LABELS_KEY] == HOLDOUT_CT).values
adata.obs["is_holdout"] = is_crc & is_mye
keep = ~adata.obs["is_holdout"].values
n_keep = int(keep.sum())

ln = sc.AnnData(adata.layers["counts"].copy())
sc.pp.normalize_total(ln, target_sum=TARGET_SUM)
sc.pp.log1p(ln)
lognorm = np.asarray(ln.X.todense() if sp.issparse(ln.X) else ln.X, dtype="float32")
del ln

sq_norm = (lognorm ** 2).sum(1, keepdims=True)
sq_norm[sq_norm == 0] = 1.0
x_released = lognorm / sq_norm            # datareader.py:71-73, verbatim

print(f"held out (Myeloid, CRC) = {int(adata.obs.is_holdout.sum())} cells")
print(f"graph nodes kept        = {n_keep} cells x {adata.n_vars} genes")

held out (Myeloid, CRC) = 5035 cells
graph nodes kept        = 106954 cells x 2000 genes


## 3. Celcomen's released training loop, at this scale

`celcomen.training_plan.train.train` is called unchanged. Only the edge list is built directly with
`kneighbors_graph` instead of through `get_dataset_loaders`, because that dataloader computes
`squareform(pdist(coords))` unconditionally - a dense cell-by-cell float64 array, i.e. 42.6 GiB of
host RAM for the condensed `pdist` alone and 85.2 GiB for its `squareform` at this cell count, before
training even starts. Bypassing it gives the training loop the best possible chance.

In [4]:
# --- Released train(), unchanged ------------------------------------------
import torch_geometric
from torch_geometric.loader import DataLoader

from celcomen.models.celcomen import celcomen
from celcomen.training_plan.train import train
from celcomen.utils.helpers import normalize_g2g

n_genes = adata.n_vars
coords_keep = adata.obsm["spatial"][keep]
edge_index = torch.from_numpy(
    np.array(kneighbors_graph(coords_keep, K_NN, include_self=False).nonzero())
).long()

g2g_init = np.random.uniform(size=(n_genes, n_genes)).astype("float32")
g2g_init = normalize_g2g((g2g_init + g2g_init.T) / 2)
model = celcomen(input_dim=n_genes, output_dim=n_genes, n_neighbors=K_NN, seed=SEED)
model.set_g2g(torch.from_numpy(g2g_init.copy()))
model.set_g2g_intra(torch.from_numpy(g2g_init.copy()))
model.to(DEVICE)

data = torch_geometric.data.Data(
    x=torch.from_numpy(x_released[keep]), pos=torch.from_numpy(coords_keep),
    y=torch.Tensor([0]), edge_index=edge_index,
)
loader = DataLoader([data], batch_size=1, shuffle=True)

print(f"cells={n_keep}  genes={n_genes}  edges={edge_index.shape[1]}")
print(f"trace(msg @ gex.T) is a dense {n_keep} x {n_keep} float32 matrix "
      f"= {n_keep ** 2 * 4 / 2 ** 30:.2f} GiB\n")

try:
    losses = train(EPOCHS, LR, model, loader, zmft_scalar=ZMFT, seed=SEED, device=DEVICE)
    print(f"trained: loss {losses[0]:.3e} -> {losses[-1]:.3e}")
except Exception:
    print(traceback.format_exc()[-1400:])
finally:
    torch.cuda.empty_cache()

/data/ddimitrov/software/miniforge3/envs/celcomen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cells=106954  genes=2000  edges=641724
trace(msg @ gex.T) is a dense 106954 x 106954 float32 matrix = 42.61 GiB



  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/tmp/ipykernel_1521925/1849252038.py", line 33, in <module>
    losses = train(EPOCHS, LR, model, loader, zmft_scalar=ZMFT, seed=SEED, device=DEVICE)
  File "/data/ddimitrov/software/miniforge3/envs/celcomen/lib/python3.10/site-packages/celcomen/training_plan/train.py", line 60, in train
    loss = -(-log_z_mft + zmft_scalar * torch.trace(torch.mm(msg, torch.t(model.gex))) + zmft_scalar * torch.trace(torch.mm(msg_intra, torch.t(model.gex))) )
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 42.62 GiB. GPU 1 has a total capacity of 23.65 GiB of which 15.30 GiB is free. Process 1507619 has 728.00 MiB memory in use. Process 1507789 has 748.00 MiB memory in use. Including non-PyTorch memory, this process has 6.89 GiB memory in use. Of the allocated memory 2.44 GiB is allocated by PyTorch, and 4.00 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segm

## Conclusion

Celcomen, run as released on the slide our benchmark uses, cannot be trained: its objective
requires a dense cell-by-cell matrix, `42.62 GiB` in float32 at `106,954` cells, against 24 GiB of
device memory. The requirement grows quadratically with cell count, so it is not a matter of a
slightly larger GPU - and `crc_232` is the *smallest* slide in our cohort. Reaching the perturbation
step (Simcomen) is therefore not possible here either.

Two further notes recorded while implementing this, neither fatal to the method but relevant to
anyone reproducing it:

- `get_dataset_loaders` computes `squareform(pdist(coords))` for every sample regardless of whether
  its `distance` branch is used, and then densifies the kNN graph with `.toarray()`; both are
  cell-by-cell arrays (42.6 GiB and 85.2 GiB here). It also hard-codes `n_neighbors = 6`
  immediately after accepting it as an argument.
- The released dataloader row-normalises by `||x||**2` while the tutorial's Simcomen cell uses
  `||x||`, so training and relaxation would run on differently scaled inputs.